# FloodShift 04G — FINAL Segmentation Attempt

**This is the last segmentation optimization experiment.**

04G does **no training**. It uses the already-frozen 04E and 04F checkpoints and searches only a small,
predeclared set of validation-only fusion rules to test whether 04F's temporary-flood gain can be recovered
without sacrificing 04E's permanent-water strength.

**Stop rule:** if the validation gate is not passed, **04E is finalized immediately** and the project moves
to the next FloodShift pipeline stage. The frozen geographic test is never opened on a failed 04G.

In [ ]:
# ============================================================
# CELL: P9-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P8-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P9-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P8-C01.")

In [2]:
# ============================================================
# CELL: P9-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
# Use one T4 deliberately even if Kaggle exposes two.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, re, json, time, math, random, copy, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
from scipy import ndimage
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import rasterio
from rasterio.windows import Window

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_04G_Final_Fusion")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"
BOOTSTRAP_ROOT = OUTPUT_ROOT / "bootstrap"
for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, CALIBRATION_ROOT, BOOTSTRAP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P9-C01 — 04G FINAL-FUSION ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available(): print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


P9-C01 — 04G FINAL-FUSION ENVIRONMENT
Python      : 3.12.13
NumPy       : 1.26.4
Pandas      : 2.3.3
SciPy       : 1.16.3
PyTorch     : 2.10.0+cu128
TerraTorch  : unknown
Device      : cuda:0
Visible GPUs: 1
GPU         : Tesla T4
AMP         : True
Output root : /kaggle/working/FloodShift_04G_Final_Fusion


In [3]:
# ============================================================
# CELL: P9-C02 — LOCATE FROZEN PREPROCESSING + 04E + 04F
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)
    if cache_key in _file_cache:
        return _file_cache[cache_key]
    roots = [INPUT_ROOT, BOOTSTRAP_ROOT]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input or bootstrap."
            )
        return None
    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[cache_key] = selected
    return selected

# Auto-extract the two small experiment archives when attached.
for archive_name in ["FloodShift_FINAL_ARCHIVE.zip", "FloodShift_04F_ARCHIVE.zip"]:
    matches = list(INPUT_ROOT.rglob(archive_name))
    if matches:
        archive = sorted(matches, key=lambda p: len(str(p)))[0]
        print("Extracting:", archive)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(BOOTSTRAP_ROOT)

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04E = locate_by_saved_path("FloodShift_FINAL_TerraMind_RIFT_best.pt")
CHECKPOINT_04F = locate_by_saved_path("FloodShift_04F_best.pt")

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)
assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P9-C02 — INPUTS FOUND")
print("=" * 112)
for p in [
    FINAL_MANIFEST_PATH, GLOBAL_LOCK_PATH, OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH, CHECKPOINT_04E, CHECKPOINT_04F
]:
    print(" ", p)
print("Lock:", preprocessing_lock["status"])
print("P9-C02 COMPLETE ✅")

P9-C02 — INPUTS FOUND
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_manifest_v1.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_GLOBAL_LOCK_v1.json
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/optical/optical_10band_train_normalization_exact.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/train_raw_sar_scene_statistics.csv
  /kaggle/input/datasets/himavarshini2108/4eptfile/FloodShift_FINAL_TerraMind_RIFT_best.pt
  /kaggle/input/datasets/himavarshini2108/4fptnew/FloodShift_04F_best.pt
Lock: PERMANENTLY_FROZEN
P9-C02 COMPLETE ✅


In [5]:
# ============================================================
# CELL: P9-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P9-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

P9-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : frozen continuous P(clear)
Primary SAR input  : raw VV/VH


In [6]:
# ============================================================
# CELL: P9-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [7]:
# ============================================================
# CELL: P9-C05 — EVAL-ONLY LOADERS
# No training dataset, no crop-bank scan, no optimizer.
# ============================================================
BATCH_SIZE = 2
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

val_dataset = FixedEvalDataset(val_df, train_mode=False)
test_dataset = FixedEvalDataset(test_df, train_mode=False)

common = dict(
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common
)

batch = next(iter(val_loader))
print("=" * 112)
print("P9-C05 — EVAL BATCH CHECK")
print("=" * 112)
print("Validation tiles :", len(val_dataset))
print("Frozen test tiles:", len(test_dataset))
print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
assert batch["sar"].shape[1:] == (2, 256, 256)
assert batch["optical"].shape[1:] == (10, 256, 256)
print("P9-C05 COMPLETE ✅")

P9-C05 — EVAL BATCH CHECK
Validation tiles : 272
Frozen test tiles: 232
SAR        : (2, 2, 256, 256)
Optical    : (2, 10, 256, 256)
Reliability: (2, 1, 256, 256)
Target     : (2, 256, 256)
P9-C05 COMPLETE ✅


In [9]:
# ============================================================
# CELL: P9-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [10]:
# ============================================================
# CELL: P9-C07 — RECONSTRUCT AND LOAD 04E + 04F
# ============================================================
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_04F_SHA256 = "75f787419da48c8ce660a0ffafe5a7835c420ec9cd8a06f1d9623e3012397aac"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

sha04e = sha256_file(CHECKPOINT_04E)
sha04f = sha256_file(CHECKPOINT_04F)
print("04E SHA256:", sha04e)
print("04F SHA256:", sha04f)
assert sha04e == EXPECTED_04E_SHA256, "Wrong 04E checkpoint."
if sha04f != EXPECTED_04F_SHA256:
    print("⚠️ 04F hash differs from the run inspected earlier; state/metric audits remain mandatory.")

class WaterTypeResidualRefiner(nn.Module):
    def __init__(self, in_channels=16, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_logits, sar, optical, reliability):
        base_prob = torch.softmax(base_logits.float(), dim=1).to(base_logits.dtype)
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        return self.net(x)

class FloodShift04F(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.refiner = WaterTypeResidualRefiner()

    @property
    def backbone(self):
        return self.base.backbone

    def forward(self, sar, optical, reliability, return_aux=False):
        base_out = self.base(sar, optical, reliability, return_aux=True)
        base_logits = base_out["logits"]
        correction = self.refiner(base_logits, sar, optical, reliability)
        water_delta = correction[:, 0:1]
        type_delta = correction[:, 1:2]

        logits = base_logits.clone()
        logits[:, 0:1] = logits[:, 0:1] - 0.50 * water_delta
        logits[:, 1:2] = logits[:, 1:2] + 0.50 * water_delta - 0.50 * type_delta
        logits[:, 2:3] = logits[:, 2:3] + 0.50 * water_delta + 0.50 * type_delta

        if return_aux:
            return {
                "logits": logits,
                "base_logits": base_logits,
                "aux_logits": base_out["aux_logits"],
                "router_weights": base_out["router_weights"],
                "correction": correction,
            }
        return logits

def sanitize_known_unused(state):
    state = dict(state)
    for key in ["backbone.missing_mod_token", "base.backbone.missing_mod_token"]:
        if key in state and torch.is_floating_point(state[key]) and not torch.isfinite(state[key]).all():
            state[key] = torch.zeros_like(state[key])
    return state

def load_04e_as_wrapper(path):
    base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["state_dict"] if "state_dict" in payload else payload
    state = sanitize_known_unused(state)
    missing, unexpected = base.load_state_dict(state, strict=False)
    assert not missing, f"04E missing keys: {missing[:10]}"
    assert not unexpected, f"04E unexpected keys: {unexpected[:10]}"
    return FloodShift04F(base)

def load_04f(path):
    base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
    model = FloodShift04F(base)
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["state_dict"] if "state_dict" in payload else payload
    state = sanitize_known_unused(state)
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not missing, f"04F missing keys: {missing[:10]}"
    assert not unexpected, f"04F unexpected keys: {unexpected[:10]}"
    return model

model_04e = load_04e_as_wrapper(CHECKPOINT_04E).to(DEVICE).eval()
model_04f = load_04f(CHECKPOINT_04F).to(DEVICE).eval()

# 04E wrapper must be an exact identity around the frozen base.
with torch.inference_mode():
    sar = batch["sar"][:1].to(DEVICE)
    opt = batch["optical"][:1].to(DEVICE)
    rel = batch["reliability"][:1].to(DEVICE)
    out = model_04e(sar, opt, rel, return_aux=True)
    diff = (out["logits"] - out["base_logits"]).abs().max().item()
print("04E wrapper identity max diff:", f"{diff:.3e}")
assert diff == 0.0

for name, mdl in [("04E", model_04e), ("04F", model_04f)]:
    bad = [
        n for n, t in mdl.state_dict().items()
        if torch.is_floating_point(t) and not torch.isfinite(t).all()
    ]
    assert not bad, f"{name} non-finite tensors: {bad[:10]}"

print("P9-C07 COMPLETE ✅")

04E SHA256: 6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822
04F SHA256: 75f787419da48c8ce660a0ffafe5a7835c420ec9cd8a06f1d9623e3012397aac


INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:Loaded weights for RED_EDGE_3 in position 5 of patch embed
INFO:root:Loaded weights for NIR_BROAD in position 6 of patch embed
INFO:root:Loaded weights for NIR_NARROW in position 7 of patch embed
INFO:root:Loaded weights for SWIR_1 in position 8 of patch embed
INFO:root:Loaded weights for SWIR_2 in position 9 of patch embed
INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:

04E wrapper identity max diff: 0.000e+00
P9-C07 COMPLETE ✅


In [11]:
# ============================================================
# CELL: P9-C08 — METRICS + 4-WAY TTA LOGITS
# ============================================================
def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()
    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices, minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    den_iou = tp + fp + fn
    iou = np.divide(tp, den_iou, out=np.full_like(tp, np.nan), where=den_iou > 0)
    den_dice = 2 * tp + fp + fn
    dice = np.divide(2 * tp, den_dice, out=np.full_like(tp, np.nan), where=den_dice > 0)

    out = {
        "mIoU": float(np.nanmean(iou)),
        "macro_dice": float(np.nanmean(dice)),
        "confusion": cm.tolist(),
    }
    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])
        out[f"{name}_Dice"] = float(dice[i])
    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(tensor, mode):
    if mode == "identity":
        return tensor
    if mode == "hflip":
        return torch.flip(tensor, dims=[-1])
    if mode == "vflip":
        return torch.flip(tensor, dims=[-2])
    if mode == "rot180":
        return torch.rot90(tensor, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(tensor, mode):
    return _tta_transform(tensor, mode)

@torch.inference_mode()
def tta_logits_batch(model, sar, optical, reliability):
    modes = ["identity", "hflip", "vflip", "rot180"]
    total = None
    for mode in modes:
        sar_t = _tta_transform(sar, mode)
        opt_t = _tta_transform(optical, mode)
        rel_t = _tta_transform(reliability, mode)
        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            logits = model(sar_t, opt_t, rel_t, return_aux=False)
        logits = _tta_inverse(logits, mode).float()
        total = logits if total is None else total + logits
    return total / len(modes)

print("P9-C08 COMPLETE ✅")

P9-C08 COMPLETE ✅


In [12]:
# ============================================================
# CELL: P9-C09 — CACHE 04E + 04F VALIDATION TTA LOGITS
# This is the expensive part. Everything after this is fast.
# ============================================================
val_cache = []
cm_e = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
cm_f = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

start = time.time()
for bi, b in enumerate(val_loader, start=1):
    sar = b["sar"].to(DEVICE, non_blocking=True)
    opt = b["optical"].to(DEVICE, non_blocking=True)
    rel = b["reliability"].to(DEVICE, non_blocking=True)
    target_gpu = b["target"].to(DEVICE, non_blocking=True)

    le = tta_logits_batch(model_04e, sar, opt, rel)
    lf = tta_logits_batch(model_04f, sar, opt, rel)

    cm_e += confusion_from_predictions(le.argmax(1), target_gpu).cpu()
    cm_f += confusion_from_predictions(lf.argmax(1), target_gpu).cpu()

    val_cache.append({
        "e": le.half().cpu(),
        "f": lf.half().cpu(),
        "y": b["target"].to(torch.uint8).cpu(),
    })

    if bi == 1 or bi % 20 == 0 or bi == len(val_loader):
        print(f"[{bi:03d}/{len(val_loader)}] validation cache")

metrics_e = metrics_from_confusion(cm_e)
metrics_f = metrics_from_confusion(cm_f)
print("\n04E 4-way TTA:", compact_metrics(metrics_e))
print("04F 4-way TTA:", compact_metrics(metrics_f))
print("Cache minutes:", round((time.time() - start) / 60, 1))

# Mandatory reproduction guardrails.
assert abs(metrics_e["mIoU"] - 0.7791) < 0.010, "04E validation reproduction failed."
assert abs(metrics_f["mIoU"] - 0.7789) < 0.010, "04F validation reproduction failed."
print("P9-C09 COMPLETE ✅")

[001/136] validation cache
[020/136] validation cache
[040/136] validation cache
[060/136] validation cache
[080/136] validation cache
[100/136] validation cache
[120/136] validation cache
[136/136] validation cache

04E 4-way TTA: {'mIoU': 0.7791, 'non_flood': 0.9713, 'permanent': 0.8244, 'temporary': 0.5416}
04F 4-way TTA: {'mIoU': 0.7789, 'non_flood': 0.9765, 'permanent': 0.7855, 'temporary': 0.5747}
Cache minutes: 1.1
P9-C09 COMPLETE ✅


In [13]:
# ============================================================
# CELL: P9-C10 — VALIDATION-ONLY FUSION SEARCH
#
# Two deliberately small families:
# A) hierarchical probability fusion
# B) selective 04F -> temporary override on 04E disagreements
#
# No model weights are trained.
# ============================================================
rng = np.random.default_rng(SEED)
sample_e, sample_f, sample_y = [], [], []

# Approximately uniform random valid-pixel sample across batches.
SAMPLE_PER_BATCH = 4096
for item in val_cache:
    y = item["y"].numpy().reshape(-1)
    valid_idx = np.flatnonzero(y != IGNORE_INDEX)
    if valid_idx.size > SAMPLE_PER_BATCH:
        idx = rng.choice(valid_idx, size=SAMPLE_PER_BATCH, replace=False)
    else:
        idx = valid_idx

    e = item["e"].float().permute(0, 2, 3, 1).reshape(-1, 3).numpy()[idx]
    f = item["f"].float().permute(0, 2, 3, 1).reshape(-1, 3).numpy()[idx]
    sample_e.append(e)
    sample_f.append(f)
    sample_y.append(y[idx].astype(np.int64))

sample_e = np.concatenate(sample_e, axis=0)
sample_f = np.concatenate(sample_f, axis=0)
sample_y = np.concatenate(sample_y, axis=0)

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)

pe = softmax_np(sample_e)
pf = softmax_np(sample_f)
pred_e = pe.argmax(1)
pred_f = pf.argmax(1)

def sample_metrics(pred):
    cm = np.bincount(
        NUM_CLASSES * sample_y + pred.astype(np.int64),
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)
    return metrics_from_confusion(cm)

def hierarchical_pred(pe, pf, aw, bt, bias):
    # Separate water-vs-background from permanent-vs-temporary.
    pwe = np.clip(pe[:, 1] + pe[:, 2], 1e-6, 1 - 1e-6)
    pwf = np.clip(pf[:, 1] + pf[:, 2], 1e-6, 1 - 1e-6)
    pw = (1.0 - aw) * pwe + aw * pwf

    odds_e = np.log(np.clip(pe[:, 2], 1e-7, 1.0)) - np.log(np.clip(pe[:, 1], 1e-7, 1.0))
    odds_f = np.log(np.clip(pf[:, 2], 1e-7, 1.0)) - np.log(np.clip(pf[:, 1], 1e-7, 1.0))
    type_logit = (1.0 - bt) * odds_e + bt * odds_f + bias
    ptw = 1.0 / (1.0 + np.exp(-np.clip(type_logit, -30, 30)))

    p0 = 1.0 - pw
    p1 = pw * (1.0 - ptw)
    p2 = pw * ptw
    return np.stack([p0, p1, p2], axis=1).argmax(1)

def override_pred(pe, pf, mode, conf, margin):
    base = pe.argmax(1).copy()
    pf_pred = pf.argmax(1)
    temp_conf = pf[:, 2]
    temp_margin = pf[:, 2] - np.maximum(pf[:, 0], pf[:, 1])

    if mode == "perm_only":
        eligible = (base == 1) & (pf_pred == 2)
    else:
        eligible = (base != 2) & (pf_pred == 2)

    switch = eligible & (temp_conf >= conf) & (temp_margin >= margin)
    base[switch] = 2
    return base

candidates = [
    {"family": "baseline", "name": "04E"},
    {"family": "baseline", "name": "04F"},
]

for aw in [0.00, 0.15, 0.30, 0.45]:
    for bt in [0.00, 0.25, 0.50, 0.75, 1.00]:
        for bias in [-0.30, -0.20, -0.10, 0.00, 0.10, 0.20, 0.30]:
            candidates.append({
                "family": "hierarchical",
                "aw": aw, "bt": bt, "bias": bias
            })

for mode in ["perm_only", "any_non_temp"]:
    for conf in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
        for margin in [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]:
            candidates.append({
                "family": "override",
                "mode": mode, "conf": conf, "margin": margin
            })

scored = []
for c in candidates:
    if c["family"] == "baseline":
        pred = pred_e if c["name"] == "04E" else pred_f
    elif c["family"] == "hierarchical":
        pred = hierarchical_pred(pe, pf, c["aw"], c["bt"], c["bias"])
    else:
        pred = override_pred(pe, pf, c["mode"], c["conf"], c["margin"])

    m = sample_metrics(pred)
    scored.append({**c, **{
        "sample_mIoU": m["mIoU"],
        "sample_perm": m["permanent_water_IoU"],
        "sample_temp": m["temporary_flood_IoU"],
    }})

# Carry forward a compact but diverse set for exact full-validation scoring.
rank_miou = sorted(scored, key=lambda x: x["sample_mIoU"], reverse=True)[:8]
rank_gateish = sorted(
    scored,
    key=lambda x: (
        x["sample_perm"] >= 0.80,
        x["sample_temp"] >= 0.55,
        x["sample_mIoU"]
    ),
    reverse=True
)[:8]
rank_temp = sorted(
    [x for x in scored if x["sample_perm"] >= 0.80],
    key=lambda x: (x["sample_temp"], x["sample_mIoU"]),
    reverse=True
)[:4]

shortlist = []
seen = set()
for c in rank_miou + rank_gateish + rank_temp + [
    {"family": "baseline", "name": "04E"},
    {"family": "baseline", "name": "04F"},
]:
    key = json.dumps({k: v for k, v in c.items() if not k.startswith("sample_")}, sort_keys=True)
    if key not in seen:
        seen.add(key)
        shortlist.append({k: v for k, v in c.items() if not k.startswith("sample_")})

print("Sample pixels:", len(sample_y))
print("Total candidates searched:", len(scored))
print("Full-validation shortlist:", len(shortlist))
print("Top sampled candidates:")
for x in sorted(scored, key=lambda z: z["sample_mIoU"], reverse=True)[:8]:
    print({k: (round(v,4) if isinstance(v,float) else v) for k,v in x.items()})
print("P9-C10 COMPLETE ✅")

Sample pixels: 557056
Total candidates searched: 250
Full-validation shortlist: 10
Top sampled candidates:
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.3, 'sample_mIoU': 0.795, 'sample_perm': 0.8288, 'sample_temp': 0.5807}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.2, 'sample_mIoU': 0.7941, 'sample_perm': 0.8269, 'sample_temp': 0.5802}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.25, 'bias': -0.3, 'sample_mIoU': 0.7934, 'sample_perm': 0.8244, 'sample_temp': 0.5804}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.1, 'sample_mIoU': 0.7929, 'sample_perm': 0.8242, 'sample_temp': 0.5793}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.25, 'bias': -0.2, 'sample_mIoU': 0.7919, 'sample_perm': 0.8215, 'sample_temp': 0.5789}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': 0.0, 'sample_mIoU': 0.7913, 'sample_perm': 0.8207, 'sample_temp': 0.5778}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.5, 'bias': -0.3, 'sample_mIoU': 0.791, 'sample_p

In [14]:
# ============================================================
# CELL: P9-C11 — EXACT FULL-VALIDATION SCORING + FINAL GATE
# ============================================================
PROMOTION_MIN_MIOU = 0.790
PROMOTION_MIN_TEMP_IOU = 0.570
PROMOTION_MIN_PERM_IOU = 0.820

def apply_candidate_torch(le, lf, c):
    pe = torch.softmax(le.float(), dim=1)
    pf = torch.softmax(lf.float(), dim=1)

    if c["family"] == "baseline":
        return pe.argmax(1) if c["name"] == "04E" else pf.argmax(1)

    if c["family"] == "hierarchical":
        aw, bt, bias = float(c["aw"]), float(c["bt"]), float(c["bias"])
        pwe = (pe[:, 1] + pe[:, 2]).clamp(1e-6, 1 - 1e-6)
        pwf = (pf[:, 1] + pf[:, 2]).clamp(1e-6, 1 - 1e-6)
        pw = (1.0 - aw) * pwe + aw * pwf

        odds_e = torch.log(pe[:, 2].clamp_min(1e-7)) - torch.log(pe[:, 1].clamp_min(1e-7))
        odds_f = torch.log(pf[:, 2].clamp_min(1e-7)) - torch.log(pf[:, 1].clamp_min(1e-7))
        ptw = torch.sigmoid((1.0 - bt) * odds_e + bt * odds_f + bias)

        probs = torch.stack([
            1.0 - pw,
            pw * (1.0 - ptw),
            pw * ptw,
        ], dim=1)
        return probs.argmax(1)

    base = pe.argmax(1)
    pf_pred = pf.argmax(1)
    temp_conf = pf[:, 2]
    temp_margin = pf[:, 2] - torch.maximum(pf[:, 0], pf[:, 1])

    if c["mode"] == "perm_only":
        eligible = (base == 1) & (pf_pred == 2)
    else:
        eligible = (base != 2) & (pf_pred == 2)

    switch = (
        eligible
        & (temp_conf >= float(c["conf"]))
        & (temp_margin >= float(c["margin"]))
    )
    out = base.clone()
    out[switch] = 2
    return out

cms = [
    torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    for _ in shortlist
]

for item in val_cache:
    le = item["e"].to(DEVICE)
    lf = item["f"].to(DEVICE)
    y = item["y"].long().to(DEVICE)

    for i, c in enumerate(shortlist):
        pred = apply_candidate_torch(le, lf, c)
        cms[i] += confusion_from_predictions(pred, y).cpu()

full_results = []
for c, cm in zip(shortlist, cms):
    m = metrics_from_confusion(cm)
    full_results.append({
        "config": c,
        "metrics": m,
        "compact": compact_metrics(m),
    })

full_results = sorted(
    full_results,
    key=lambda x: x["metrics"]["mIoU"],
    reverse=True
)

print("=" * 112)
print("P9-C11 — FULL VALIDATION RESULTS")
print("=" * 112)
for row in full_results[:12]:
    print(row["config"], "->", row["compact"])

eligible = [
    x for x in full_results
    if (
        x["metrics"]["mIoU"] >= PROMOTION_MIN_MIOU
        and x["metrics"]["temporary_flood_IoU"] >= PROMOTION_MIN_TEMP_IOU
        and x["metrics"]["permanent_water_IoU"] >= PROMOTION_MIN_PERM_IOU
    )
]

if eligible:
    winner = eligible[0]
    promotion_pass = True
    print("\nPROMOTION: PASS ✅")
    print("04G winner:", winner["config"])
    print("Validation:", winner["compact"])
else:
    promotion_pass = False
    winner = {
        "config": {"family": "baseline", "name": "04E"},
        "metrics": metrics_e,
        "compact": compact_metrics(metrics_e),
    }
    print("\nPROMOTION: FAIL ❌")
    print("STOP RULE TRIGGERED: 04E IS FINALIZED.")
    print("Official validation reference:", winner["compact"])

print(
    f"Gate: mIoU>={PROMOTION_MIN_MIOU:.3f}, "
    f"temp>={PROMOTION_MIN_TEMP_IOU:.3f}, "
    f"perm>={PROMOTION_MIN_PERM_IOU:.3f}"
)

P9-C11 — FULL VALIDATION RESULTS
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.3} -> {'mIoU': 0.7953, 'non_flood': 0.9749, 'permanent': 0.8339, 'temporary': 0.5772}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.2} -> {'mIoU': 0.7941, 'non_flood': 0.9748, 'permanent': 0.8312, 'temporary': 0.5762}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.25, 'bias': -0.3} -> {'mIoU': 0.7934, 'non_flood': 0.9748, 'permanent': 0.8293, 'temporary': 0.5763}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': -0.1} -> {'mIoU': 0.7928, 'non_flood': 0.9748, 'permanent': 0.8283, 'temporary': 0.5751}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.25, 'bias': -0.2} -> {'mIoU': 0.7918, 'non_flood': 0.9748, 'permanent': 0.8258, 'temporary': 0.5748}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.0, 'bias': 0.0} -> {'mIoU': 0.7913, 'non_flood': 0.9748, 'permanent': 0.8252, 'temporary': 0.5739}
{'family': 'hierarchical', 'aw': 0.45, 'bt': 0.5, 'bias': -0.3} -> {'mIoU': 0.7904, 

In [15]:
# ============================================================
# CELL: P9-C12 — FROZEN TEST, ONLY IF 04G PASSES
# ============================================================
# If C11 fails: DO NOT change this flag. 04E is final.
# If C11 passes: set True and rerun ONLY this cell once.
UNLOCK_FROZEN_TEST = False

test_metrics = None

if not promotion_pass:
    print("🔒 04G failed validation gate. Test stays locked. 04E is FINAL.")
elif not UNLOCK_FROZEN_TEST:
    print("✅ 04G passed validation, but test remains manually locked.")
    print("Set UNLOCK_FROZEN_TEST=True and rerun ONLY P9-C12.")
else:
    print("Running ONE frozen Pakistan+Spain test evaluation...")
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

    for bi, b in enumerate(test_loader, start=1):
        sar = b["sar"].to(DEVICE, non_blocking=True)
        opt = b["optical"].to(DEVICE, non_blocking=True)
        rel = b["reliability"].to(DEVICE, non_blocking=True)
        y = b["target"].to(DEVICE, non_blocking=True)

        le = tta_logits_batch(model_04e, sar, opt, rel)
        lf = tta_logits_batch(model_04f, sar, opt, rel)
        pred = apply_candidate_torch(le, lf, winner["config"])
        cm += confusion_from_predictions(pred, y).cpu()

        if bi == 1 or bi % 20 == 0 or bi == len(test_loader):
            print(f"[{bi:03d}/{len(test_loader)}] frozen test")

    test_metrics = metrics_from_confusion(cm)

    print("=" * 112)
    print("04G FROZEN GEOGRAPHIC TEST")
    print("=" * 112)
    print(compact_metrics(test_metrics))
    print("04E official test reference:",
          {"mIoU": 0.7162, "non_flood": 0.9628, "permanent": 0.6183, "temporary": 0.5675})

✅ 04G passed validation, but test remains manually locked.
Set UNLOCK_FROZEN_TEST=True and rerun ONLY P9-C12.


In [16]:
# ============================================================
# CELL: P9-C13 — EXPORT DECISION RECORD
# ============================================================
record = {
    "experiment": "FloodShift-04G-Final-Validation-Fusion",
    "purpose": "last segmentation experiment; parameter-free validation-only fusion",
    "checkpoints": {
        "04E": str(CHECKPOINT_04E),
        "04E_sha256": sha04e,
        "04F": str(CHECKPOINT_04F),
        "04F_sha256": sha04f,
    },
    "validation": {
        "04E_4way_tta": metrics_e,
        "04F_4way_tta": metrics_f,
        "shortlist_results": full_results,
        "gate": {
            "mIoU": PROMOTION_MIN_MIOU,
            "temporary": PROMOTION_MIN_TEMP_IOU,
            "permanent": PROMOTION_MIN_PERM_IOU,
        },
        "promotion_pass": bool(promotion_pass),
        "winner": winner,
    },
    "test_metrics": test_metrics,
    "stop_rule": (
        "If 04G fails validation promotion, 04E is finalized and "
        "segmentation optimization ends."
    ),
}
out_json = OUTPUT_ROOT / "FloodShift_04G_decision_record.json"
with open(out_json, "w") as f:
    json.dump(record, f, indent=2)

print("Saved:", out_json)
print("FINAL SEGMENTATION DECISION:",
      "04G promoted" if promotion_pass else "04E FINALIZED")
print("P9-C13 COMPLETE ✅")

Saved: /kaggle/working/FloodShift_04G_Final_Fusion/FloodShift_04G_decision_record.json
FINAL SEGMENTATION DECISION: 04G promoted
P9-C13 COMPLETE ✅


## End condition

- **04G gate passes:** manually unlock C12 once, record the frozen test, then close segmentation.
- **04G gate fails:** do not run C12. **04E is the final segmentation model.**

No 04H.